# Benchmark truy xuất văn bản pháp luật Việt Nam

**Vị trí trong luồng MLOps:** notebook `legal_rag_build` dựng index + bộ câu hỏi chuẩn và chấm **một** cấu hình
(production) để làm cổng chất lượng. Notebook này dùng **đúng index và đúng bộ câu hỏi đó** để so **nhiều phương pháp**
— chọn phương pháp/tham số cho production — và ghi tất cả vào MLflow (experiment **legal-rag-benchmark**).

```
legal_rag_build (Output: out/index + out/golden_vK.jsonl.gz) ──Add Input──► notebook này
   ─► 8 phương pháp, tham số dò trên dev, chấm trên test, kiểm định Holm ─► dashboard.html
   ─► MLflow: run cha (dashboard, đường nDCG theo α / β) + mỗi phương pháp một run con (đủ chỉ số, đường Recall@k)
Thấy phương pháp tốt hơn production một cách có ý nghĩa (cột sig) ─► sửa Config / rerank.enabled ─► build lại ─► cổng
```
**Code:** lấy từ repo (clone lúc chạy) — `mlops/bench_lib.py` (= legal_eval.py cũ; phần chung dùng lại legal_core,
pipeline, mlops.labels, đã đối chiếu giống hệt) và `mlops/bench_dashboard.py`. Không còn `%%writefile` trong notebook.
**Bộ câu hỏi:** đọc `golden_vK.jsonl.gz` của notebook build (cùng chia dev/test seed 42) thay vì tự tạo lại —
nên điểm ở đây so được trực tiếp với cổng chất lượng.

| Nhóm | Phương pháp (mỗi dòng là một "run") |
|---|---|
| **Lexical** | TF-IDF · BM25 tách âm tiết · BM25 không bỏ stopword · BM25 âm tiết+bigram · BM25 tách từ (pyvi) — dò k1, b trên dev |
| **Learned sparse** | bge-m3 lexical weights |
| **Dense** | AITeamVN/Vietnamese_Embedding · bge-m3 gốc · *(model fine-tune của bạn — điền `FINETUNED_MODEL`)* |
| **Biến thể chunk** (dense AITeamVN) | Điều nguyên (cắt 512 token) · theo cấu trúc (mặc định) · theo từng Khoản |
| **Hybrid** | RRF BM25+dense · tổng có trọng số α (dò trên dev) · RRF dense+sparse · RRF 3 đường |
| **Lexical trên Điều nguyên** | BM25 trên 65k Điều không chia nhỏ (BM25 không bị giới hạn 512 token) |
| **Rerank** | AITeamVN/Vietnamese_Reranker trên top-20 / 50 / 100 chunk · chỉ dùng điểm reranker, hoặc trộn với điểm hybrid (β dò trên dev) · chấm bằng câu hỏi đầy đủ hoặc chỉ tiêu đề |
| **Structured** | Câu hỏi có ghi "Điều X Luật Y" → tra thẳng Điều đó, rồi mới đến kết quả tìm kiếm |

**Metric (chấm ở cấp Điều):** Hit@k, Recall@k, Precision@k, F2@k (metric của ALQAC), R-Precision, MRR@10, nDCG@10, MAP@100 · khoảng tin cậy bootstrap · kiểm định randomization theo cặp + **hiệu chỉnh Holm** khi so nhiều phương pháp.
**Đầu ra chính:** `dashboard.html` — dashboard tương tác (mở bằng trình duyệt).

**Bản v5 (sửa sau rà soát):** đọc `date_sent` đúng kiểu ngày · hiệu chỉnh bộ lọc đáp án chỉ trên dev · lấy 1.000 chunk/nhánh (3.000 cho biến thể Khoản) · min-max có sàn khi trộn · BM25 có tần suất từ trong câu hỏi · pyvi giữ chữ hoa · BM25 trên Điều nguyên · rerank trộn điểm + rerank bằng tiêu đề · nạp model **trước** khi bấm giờ, không nhân số GPU · phương án tham chiếu cho kiểm định chọn trên **dev** · phân tích độ nhạy khi bỏ Điều chung chung.

### Chế độ LITE (mặc định) — 8 phương pháp chính
`LITE=True` ở mục 1 chỉ chấm 8 bậc: **TF-IDF → BM25 âm tiết → BM25 bigram** (truyền thống, không cần GPU) → **bge-m3 gốc → AITeamVN** (tìm theo nghĩa, chưa / đã fine-tune) → **hybrid** (trộn) → **rerank cả câu hỏi** (cách dùng ngây thơ) → **rerank tiêu đề + trộn điểm** (phương án đề xuất).
Các thí nghiệm phụ (pyvi, stopword, BM25 trên Điều, sparse, RRF, rerank 50/100, biến thể chunk, tra trích dẫn) đã có kết luận ở lần chạy đầy đủ v5 → đặt `LITE=False` để chạy lại toàn bộ 25 cấu hình.

### Chạy trên Kaggle
1. **Add Input** → Output của notebook `legal_rag_build` (có `out/index/chunks.parquet`, `embeddings.f16.npy`, `golden_v1.jsonl.gz`).
   Không có golden trong Input thì notebook tải `eval/golden_v1.jsonl.gz` từ MinIO (cần secrets S3_*), hoặc tự tạo bằng `mlops.labels`.
2. **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets** (tuỳ chọn, để ghi MLflow): `MLFLOW_TRACKING_URI`,
   `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`; repo private thì thêm `GITHUB_TOKEN`.
3. **Save Version → Save & Run All.** LITE ~30–40 phút trên 2×T4 (bge-m3 phải embedding lại cả Pháp điển ~10 phút, rerank ~6 phút).
   Phần tùy chọn nào lỗi sẽ **tự bỏ qua**, phần còn lại vẫn chạy.

In [ ]:
# Lấy code từ repo (clone vào /tmp: token không lọt vào Output) + secrets MLflow / MinIO
import os, sys, subprocess, logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
for _n in ("httpx", "httpcore", "urllib3", "sentence_transformers", "datasets", "huggingface_hub", "filelock", "transformers"):
    logging.getLogger(_n).setLevel(logging.WARNING)
REPO, BRANCH = "https://github.com/liesyx/VN-Legal-Retriever-Eval", "main"

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

for k in ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD", "S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY"]:
    v = secret(k)
    if v:
        os.environ[k] = v
SRC = "/tmp/legal_rag"
tok = secret("GITHUB_TOKEN")
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO, SRC], check=True)
COMMIT = subprocess.run(["git", "-C", SRC, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, SRC)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyvi"], check=False)    # chỉ dùng khi LITE=False
print("code:", REPO, BRANCH, COMMIT, "· MLflow:", os.environ.get("MLFLOW_TRACKING_URI") or "(không ghi)")

## 1. Cấu hình

In [ ]:
import os, re, sys, json, time, glob, math, random, gc, traceback
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch

CFG = dict(
    LITE=True,                       # True: chỉ chạy 8 phương pháp chính (bậc thang truyền thống → AI → rerank)
                                     # False: chạy đủ 25 cấu hình như bản v5 (để khám phá / phụ lục)
    LITE_RERANK_ABLATION=False,      # (LITE) thêm 2 bản rerank trung gian: chỉ đổi sang tiêu đề / chỉ trộn điểm
    QA_ID="tmquan/pbgdpl-vn-legal-qna", QA_CFG="qa",   # chỉ dùng khi không có bộ câu hỏi chuẩn (tự tạo bằng mlops.labels)
    GOLDEN_REF="eval/golden_v1.jsonl.gz",              # = quality.golden trong values.yaml (tên file trên MinIO)
    EMB_MODEL="AITeamVN/Vietnamese_Embedding", RERANK_MODEL="AITeamVN/Vietnamese_Reranker",
    BGEM3_MODEL="BAAI/bge-m3",
    FINETUNED_MODEL=None,            # vd. "your-hf-user/legal-embedding-ft" hoặc đường dẫn trong /kaggle/input
    QUERY_FIELD="question",          # "question" (sát thực tế) | "title+question" (dễ hơn)
    MIN_QUESTION_CHARS=20, DEV_FRAC=0.30, SEED=42,
    TOPN_CHUNKS=1000,                # số chunk mỗi nhánh trả về trước khi trộn / gộp về Điều (200 làm hụt ứng viên)
    TOPN_CHUNKS_KHOAN=3000,          # biến thể "theo Khoản" có ~3x số chunk -> lấy nhiều hơn để đủ 100 Điều
    BM25_QTF=["binary", "k3"],       # tần suất từ trong câu hỏi: bỏ qua | bão hoà kiểu BM25 (k3=8); dò trên dev
    RUN_BM25_ARTICLE=True,           # thêm BM25 trên Điều nguyên vẹn
    FUSION_FLOOR=0.05,               # min-max về [0.05, 1]: hạng cuối của một nhánh vẫn hơn "không có mặt"
    BM25_GRID=[(1.2, 0.75), (0.9, 0.4), (1.5, 0.75), (1.2, 0.9), (2.0, 0.75)],
    ALPHAS=[round(a, 1) for a in np.arange(0, 1.01, 0.1)],
    RUN_BGEM3=True, RUN_CHUNK_VARIANTS=True, RUN_STRUCTURED=True,
    RUN_RERANK=True, RERANK_DEPTHS=[20, 50, 100], RERANK_MAX_LEN=1024, RERANK_BATCH=32,
    RERANK_ON_DEV=True,              # chấm cả dev để dò β (trộn điểm) và để chọn phương án tham chiếu công bằng
    RERANK_BETAS=[round(b, 1) for b in np.arange(0, 1.01, 0.1)],
    RERANK_TITLE=True, RERANK_TITLE_DEPTH=50,   # thử reranker với TIÊU ĐỀ câu hỏi (ngắn) thay vì cả câu hỏi
    GENERIC_TITLE_RE=r"giải thích từ ngữ|phạm vi điều chỉnh|đối tượng áp dụng|quyền yêu cầu tòa án bảo vệ",
    RERANK_MAX_QUERIES=None,         # vd. 400 để chạy nhanh (lấy mẫu từ tập test; dev vẫn chấm đủ)
    PRIMARY="ndcg@10",
    REFERENCE_DATE="2100-01-01",
)
if CFG["LITE"]:
    # Bỏ các thí nghiệm phụ đã có kết luận ở lần chạy đầy đủ (v5): BM25 trên Điều, biến thể chunk, tra trích dẫn,
    # rerank sâu 50/100. BM25 dùng cách đếm từ k3 (dev đã chọn k3 ở bản đầy đủ).
    CFG.update(RUN_BM25_ARTICLE=False, RUN_CHUNK_VARIANTS=False, RUN_STRUCTURED=False,
               BM25_QTF=["k3"], RERANK_DEPTHS=[20], RERANK_TITLE_DEPTH=20)
    print("Chế độ LITE: TF-IDF · BM25 âm tiết · BM25 bigram · bge-m3 gốc · AITeamVN · hybrid · rerank câu hỏi · rerank tiêu đề + trộn")
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

hits = sorted(glob.glob("/kaggle/input/**/chunks.parquet", recursive=True)) or glob.glob("**/chunks.parquet", recursive=True)
if not hits:                                   # in ra Input đang có để biết thiếu gì
    print("Các file trong /kaggle/input:")
    for _p in sorted(glob.glob("/kaggle/input/**", recursive=True))[:60]:
        print("  ", _p)
assert hits, ("Không thấy chunks.parquet. Input phải là một PHIÊN BẢN đã 'Save & Run All' của legal_rag_build "
              "(Output có out/index/chunks.parquet) — Quick Save hay phiên bản lỗi thì không có Output")
INDEX_DIR = Path(hits[0]).parent
OUT_DIR = Path("/kaggle/working/benchmark") if Path("/kaggle").exists() else Path("benchmark")
CACHE = OUT_DIR / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
# Dùng lại embedding của lần chạy trước: Add Input chính output notebook benchmark cũ (có thư mục cache/)
for _f in glob.glob("/kaggle/input/**/cache/*.np[yz]", recursive=True):
    if not (CACHE / Path(_f).name).exists():
        os.symlink(_f, CACHE / Path(_f).name); print("dùng lại cache:", Path(_f).name)
N_GPU = torch.cuda.device_count()
DEVS = [f"cuda:{i}" for i in range(max(1, N_GPU))]
print("index:", INDEX_DIR, "| GPUs:", N_GPU)

def optional(name, fn):
    # Chạy một phần tùy chọn; lỗi thì in cảnh báo và đi tiếp
    t0 = time.time()
    try:
        out = fn(); print(f"✓ {name} ({time.time()-t0:.0f}s)"); return out
    except Exception as e:
        print(f"⚠️  Bỏ qua {name}: {type(e).__name__}: {e}"); traceback.print_exc(limit=2); return None

## 2. Module (từ repo)

In [ ]:
import importlib
from mlops import bench_lib as le, bench_dashboard as dash      # = legal_eval.py / bench_dashboard.py của bản cũ
from mlops import golden as gs, labels
from pipeline.index_data import article_keys
importlib.reload(le); importlib.reload(dash)
print(le.EVAL_VERSION, "· code", COMMIT)

## 3. Nạp index & bộ câu hỏi chuẩn

In [ ]:
chunks = pd.read_parquet(INDEX_DIR / "chunks.parquet")
manifest = json.loads((INDEX_DIR / "manifest.json").read_text()) if (INDEX_DIR / "manifest.json").exists() else {}
MREV = manifest.get("model", {}).get("revision")
MAXLEN = manifest.get("model", {}).get("max_seq_length", 514)
print(f"{len(chunks):,} chunk | {chunks.record_id.nunique():,} Điều | model index: {manifest.get('model', {}).get('id')}")

dieu = chunks["origin_article"].map(le.dieu_number)
rec_key = {r: (le.article_key(s, d) or f"rec:{r}") for r, s, d in zip(chunks.record_id, chunks.origin_so_hieu, dieu)}
chunk_keys = chunks.record_id.map(rec_key).to_numpy(dtype=object)
assert (chunk_keys == article_keys(chunks)).all()          # cùng cách tạo khoá Điều với production / cổng chất lượng
corpus_keys = set(chunk_keys)
key_title = dict(zip(chunk_keys, chunks.article_title))
key_cite = {k: f"{r.origin_article or ''} {r.origin_doc_type or ''} {r.origin_so_hieu or ''}".strip()
            for k, r in zip(chunk_keys, chunks.itertuples())}

art = chunks.drop_duplicates("record_id")
docs, by_name = le.build_catalog(art.source_note_text, art.origin_so_hieu, art.origin_doc_type, art.origin_effective_date)
resolver_loose = le.CitationResolver(docs, by_name)
resolver = resolver_loose          # dùng cho bước structured (đọc trích dẫn trong CÂU HỎI)
print(f"Catalog: {len(docs):,} văn bản · {len(by_name):,} tên văn bản · {len(corpus_keys):,} khoá Điều")

In [ ]:
# Bộ câu hỏi chuẩn: ưu tiên Output của notebook build -> MinIO -> tự tạo (mlops.labels, như notebook build)
_g = sorted(glob.glob("/kaggle/input/**/golden_v*.jsonl.gz", recursive=True))
_name = Path(CFG["GOLDEN_REF"]).name
GOLDEN = next((Path(p) for p in _g if Path(p).name == _name), Path(_g[-1]) if _g else None)
GOLDEN_STATS, GOLDEN_SRC = {}, ""
if GOLDEN:
    GOLDEN_SRC = f"Input: {GOLDEN.name}"
    _st = sorted(glob.glob("/kaggle/input/**/golden_stats.json", recursive=True))
    GOLDEN_STATS = json.loads(Path(_st[-1]).read_text()) if _st else {}
elif os.environ.get("S3_ACCESS_KEY"):
    import boto3
    from botocore.config import Config as _BC
    GOLDEN = OUT_DIR / _name
    boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"], aws_access_key_id=os.environ["S3_ACCESS_KEY"],
                 aws_secret_access_key=os.environ["S3_SECRET_KEY"], region_name="us-east-1",
                 config=_BC(signature_version="s3v4", s3={"addressing_style": "path"})).download_file("legal-rag", CFG["GOLDEN_REF"], str(GOLDEN))
    GOLDEN_SRC = f"MinIO: {CFG['GOLDEN_REF']}"
else:
    from datasets import load_dataset
    _qa = load_dataset(CFG["QA_ID"], CFG["QA_CFG"], split="train").to_pandas()
    _Q, GOLDEN_STATS = labels.build_golden(chunks, _qa, seed=CFG["SEED"], dev_frac=CFG["DEV_FRAC"],
                                           min_question_chars=CFG["MIN_QUESTION_CHARS"])
    _Q.to_parquet(OUT_DIR / "qrels.parquet", index=False)
    GOLDEN = OUT_DIR / _name
    gs.export(str(OUT_DIR / "qrels.parquet"), str(GOLDEN))
    GOLDEN_SRC = "tự tạo bằng mlops.labels (chưa có trên MinIO)"

Q = pd.DataFrame(gs.load(str(GOLDEN), split=None))
_miss = Q.gold.map(lambda g: not any(k in corpus_keys for k in g))
if _miss.any():
    print(f"⚠️  {_miss.sum()} câu có đáp án không còn trong index này (bộ câu hỏi dựng trên index khác) — vẫn chấm, chúng kéo điểm xuống")
if CFG["QUERY_FIELD"] == "title+question":
    Q["query"] = Q.title.fillna("") + ". " + Q["query"]
queries, gold_l = Q["query"].tolist(), Q["gold"].tolist()
is_dev = (Q.split == "dev").to_numpy(); is_test = ~is_dev
print(f"Bộ câu hỏi chuẩn ({GOLDEN_SRC}): {len(Q):,} câu · dev {is_dev.sum():,} · test {is_test.sum():,} · "
      f"có tiêu đề: {(Q.title.fillna('') != '').mean():.0%}")

## 4. Tiện ích chung

In [ ]:
from sentence_transformers import SentenceTransformer
runs, art_runs_direct, timing, info = {}, {}, {}, {}
texts = chunks["embed_text"].tolist()

def to_articles(run, keys=chunk_keys):
    return le.chunks_to_articles(run, keys, topn=100)

def subset(run, mask):
    ix, sc = run
    return [x for x, m in zip(ix, mask) if m], [x for x, m in zip(sc, mask) if m]

def score_articles(arts, mask, metric=CFG["PRIMARY"]):
    # arts: danh sách xếp hạng Điều cho ĐÚNG các câu trong mask (theo thứ tự)
    ids = np.flatnonzero(mask)
    return float(np.mean([le.per_query_metrics(arts[j], gold_l[i])[metric] for j, i in enumerate(ids)])) if len(ids) else float("nan")

def dev_score(name):
    arts = art_runs_direct.get(name) or to_articles(runs[name])
    return float(np.mean([le.per_query_metrics(arts[i], gold_l[i])[CFG["PRIMARY"]] for i in np.flatnonzero(is_dev)]))

_ST = {}
def get_st(model_id, revision, dev, max_len):
    # Nạp model một lần cho mỗi GPU rồi dùng lại -> thời gian đo không gồm lúc nạp model
    key = (model_id, revision, dev)
    if key not in _ST:
        m = SentenceTransformer(model_id, revision=revision, device=dev); m.half(); _ST[key] = m
    _ST[key].max_seq_length = max_len
    return _ST[key]

import hashlib
def fp_path(path, items, *extra):
    # Gắn "dấu vân tay" nội dung vào tên file cache: index/chunk đổi -> tên đổi -> tự tính lại, không dùng nhầm
    h = hashlib.sha1("\x1f".join(map(str, extra)).encode())
    for t in items:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x1e")
    path = Path(path)
    return path.with_name(f"{path.stem}.{h.hexdigest()[:12]}{path.suffix}")

def embed_texts(model_id, items, revision=None, max_len=MAXLEN, batch=64, cache=None):
    # Embed song song trên mọi GPU (fp16, sắp theo độ dài, tự giảm batch khi OOM); có cache .npy
    if cache:
        cache = fp_path(cache, items, model_id, revision, max_len)
    if cache and Path(cache).exists():
        E = np.load(cache)
        if len(E) == len(items):
            return E
        print(f"  cache {Path(cache).name} lệch số dòng ({len(E)} ≠ {len(items)}) → tính lại"); Path(cache).unlink()
    order = np.argsort([-len(t) for t in items])
    parts = [order[i::len(DEVS)] for i in range(len(DEVS))]
    def run(dev, idx):
        if not len(idx):
            return idx, None
        m = get_st(model_id, revision, dev, max_len)
        bs, out = batch, []
        for s in range(0, len(idx), 4096):
            part = [items[i] for i in idx[s:s + 4096]]
            while True:
                try:
                    out.append(np.asarray(m.encode(part, batch_size=bs, normalize_embeddings=True, convert_to_numpy=True,
                                                   show_progress_bar=False), dtype=np.float16)); break
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); gc.collect()
                    if bs == 1: raise
                    bs = max(1, bs // 2); print(f"  {dev}: OOM → batch {bs}")
        return idx, np.concatenate(out)
    with ThreadPoolExecutor(len(DEVS)) as ex:
        res = [r for r in ex.map(run, DEVS, parts) if r[1] is not None]
    E = np.zeros((len(items), res[0][1].shape[1]), np.float16)
    for idx, e in res:
        E[idx] = e
    if cache:
        np.save(cache, E)
    return E

def dense_search(E_np, Q_np, k=CFG["TOPN_CHUNKS"]):
    E = torch.from_numpy(np.ascontiguousarray(E_np)).to("cuda:0")
    Qt = torch.from_numpy(np.ascontiguousarray(Q_np)).to("cuda:0").to(E.dtype)
    ix_all, sc_all = [], []
    for s in range(0, len(Q_np), 256):
        sc, ix = torch.topk((Qt[s:s + 256] @ E.T).float(), min(k, E_np.shape[0]), dim=1)
        ix_all += list(ix.cpu().numpy()); sc_all += list(sc.cpu().numpy())
    del E, Qt; torch.cuda.empty_cache()
    return ix_all, sc_all

## 5. Lexical — TF-IDF & BM25 (CPU)

In [ ]:
dev_q = [q for q, d in zip(queries, is_dev) if d]
dev_g = [g for g, d in zip(gold_l, is_dev) if d]

analyzers = {"bm25_syllable": (le.syllables, 0.5), "bm25_syll_bigram": (le.syllable_bigrams, 0.5)}
if not CFG["LITE"]:
    analyzers["bm25_syllable_nostop"] = (le.syllables, 1.0)
    try:
        _pv = le.make_pyvi_analyzer(); _pv("thử nghiệm"); analyzers["bm25_word_pyvi"] = (_pv, 0.5)
    except Exception as e:
        print("Bỏ BM25 tách từ (pyvi):", repr(e))
TOK_DESC = {"bm25_syllable": "âm tiết, bỏ token xuất hiện >50% chunk (stopword tự động)",
            "bm25_syllable_nostop": "âm tiết, KHÔNG bỏ stopword",
            "bm25_syll_bigram": "âm tiết + bigram (xấp xỉ từ ghép)", "bm25_word_pyvi": "tách từ bằng pyvi"}

tf = le.TFIDF(le.syllables).fit(texts)
t1 = time.time(); runs["tfidf_syllable"] = tf.search(queries, k=CFG["TOPN_CHUNKS"])
timing["tfidf_syllable"] = (time.time() - t1) / len(queries) * 1000
info["tfidf_syllable"] = dict(desc="TF-IDF (sublinear tf) trên âm tiết, cosine"); del tf; gc.collect()
print(f"tfidf_syllable: {timing['tfidf_syllable']:.1f} ms/câu")

bm25_grid_log = []
def tune_bm25(bm, name, keys=chunk_keys):
    best_p = None
    for qtf in CFG["BM25_QTF"]:
        bm.qtf = qtf
        for k1, b in CFG["BM25_GRID"]:
            bm.set_params(k1, b)
            sc = float(np.mean([le.per_query_metrics(a, g)[CFG["PRIMARY"]]
                                for a, g in zip(to_articles(bm.search(dev_q, k=CFG["TOPN_CHUNKS"]), keys), dev_g)]))
            bm25_grid_log.append(dict(analyzer=name, qtf=qtf, k1=k1, b=b, score=sc))
            if best_p is None or sc > best_p[0]: best_p = (sc, k1, b, qtf)
    bm.qtf = best_p[3]; bm.set_params(best_p[1], best_p[2])
    return best_p

for name, (an, max_df) in analyzers.items():
    bm = le.BM25(an, max_df=max_df).fit(texts)
    best_p = tune_bm25(bm, name)
    t1 = time.time(); runs[name] = bm.search(queries, k=CFG["TOPN_CHUNKS"])
    timing[name] = (time.time() - t1) / len(queries) * 1000
    info[name] = dict(desc=f"BM25 {TOK_DESC[name]} · k1={best_p[1]}, b={best_p[2]}, qtf={best_p[3]}", vocab=len(bm.vec.vocabulary_))
    print(f"→ {name}: k1={best_p[1]} b={best_p[2]} qtf={best_p[3]} dev={best_p[0]:.4f} | vocab {info[name]['vocab']:,} | {timing[name]:.1f} ms/câu")
    del bm; gc.collect()

# BM25 trên Điều NGUYÊN VẸN (không chia chunk) — BM25 không bị giới hạn 512 token như transformer
def _bm25_article():
    c = chunks.sort_values(["record_id", "chunk_index"])
    first = c.drop_duplicates("record_id").set_index("record_id")
    bodies = c.groupby("record_id", sort=False)["body"].apply(lambda x: " ".join(x))
    rids = bodies.index.tolist()
    heads = {r: e[: len(e) - len(b)].rstrip() for r, e, b in zip(first.index, first.embed_text, first.body)}
    a_texts = [f"{heads[r]}\n{bodies[r]}" for r in rids]
    a_keys = np.array([rec_key[r] for r in rids], dtype=object)
    an_name = "bm25_syll_bigram" if "bm25_syll_bigram" in analyzers else next(iter(analyzers))
    an, max_df = analyzers[an_name]
    bm = le.BM25(an, max_df=max_df).fit(a_texts)
    name = an_name + "_article"
    best_p = tune_bm25(bm, name, a_keys)
    t1 = time.time(); r = bm.search(queries, k=CFG["TOPN_CHUNKS"])
    timing[name] = (time.time() - t1) / len(queries) * 1000
    art_runs_direct[name] = to_articles(r, a_keys)
    info[name] = dict(desc=f"BM25 {TOK_DESC[an_name]} trên {len(rids):,} Điều nguyên vẹn · k1={best_p[1]}, b={best_p[2]}, qtf={best_p[3]}")
    print(f"→ {name}: k1={best_p[1]} b={best_p[2]} qtf={best_p[3]} dev={best_p[0]:.4f} | {timing[name]:.1f} ms/câu")
    del bm; gc.collect()
if CFG["RUN_BM25_ARTICLE"]:
    optional("BM25 trên Điều nguyên vẹn", _bm25_article)

## 6. Dense — AITeamVN, bge-m3 gốc, model fine-tune

In [ ]:
E_main = np.load(INDEX_DIR / "embeddings.f16.npy"); assert E_main.shape[0] == len(chunks)
embed_texts(CFG["EMB_MODEL"], queries[:8], revision=MREV, batch=8)      # nạp model lên GPU trước, không tính giờ
try:
    _tok = get_st(CFG["EMB_MODEL"], MREV, DEVS[0], MAXLEN).tokenizer
    _lens = np.array([len(x) for x in _tok(queries, add_special_tokens=True)["input_ids"]])
    print(f"Câu hỏi dài hơn {MAXLEN} token (dense bị cắt phần sau): {(_lens > MAXLEN).sum():,}/{len(queries):,} "
          f"({(_lens > MAXLEN).mean():.0%}) · trung vị {int(np.median(_lens))} token")
except Exception as e:
    print("Không đếm được độ dài token:", repr(e))
t0 = time.time()
Qv_main = embed_texts(CFG["EMB_MODEL"], queries, revision=MREV, batch=32)
runs["dense_aiteamvn"] = dense_search(E_main, Qv_main)
timing["dense_aiteamvn"] = (time.time() - t0) / len(queries) * 1000
info["dense_aiteamvn"] = dict(desc="AITeamVN/Vietnamese_Embedding (fine-tune từ bge-m3) · vector của index")
print(f"dense_aiteamvn: {timing['dense_aiteamvn']:.1f} ms/câu")

def _finetuned():
    E = embed_texts(CFG["FINETUNED_MODEL"], texts, cache=CACHE / "E_finetuned.npy")
    embed_texts(CFG["FINETUNED_MODEL"], queries[:8], batch=8)
    t0 = time.time(); Qv = embed_texts(CFG["FINETUNED_MODEL"], queries, batch=32)
    runs["dense_finetuned"] = dense_search(E, Qv)
    timing["dense_finetuned"] = (time.time() - t0) / len(queries) * 1000
    info["dense_finetuned"] = dict(desc=f"Model fine-tune: {CFG['FINETUNED_MODEL']}")
if CFG["FINETUNED_MODEL"]:
    optional("dense_finetuned", _finetuned)

In [ ]:
_BGEM3 = {}
def _bgem3_parts(dev):
    # tokenizer, trọng số sparse và model trên từng GPU: nạp một lần, dùng lại (không tính vào thời gian đo)
    from transformers import AutoTokenizer, AutoModel
    from huggingface_hub import hf_hub_download
    if "tok" not in _BGEM3:
        _BGEM3["tok"] = AutoTokenizer.from_pretrained(CFG["BGEM3_MODEL"])
        _BGEM3["sp"] = torch.load(hf_hub_download(CFG["BGEM3_MODEL"], "sparse_linear.pt"), map_location="cpu")
    if dev not in _BGEM3:
        model = AutoModel.from_pretrained(CFG["BGEM3_MODEL"]).to(dev).half().eval()     # ép fp16 tường minh
        lin = torch.nn.Linear(model.config.hidden_size, 1)
        lin.load_state_dict({k: v.float() for k, v in _BGEM3["sp"].items()}); lin = lin.to(dev).half().eval()
        _BGEM3[dev] = (model, lin)
    return _BGEM3["tok"], _BGEM3[dev]

def bgem3_encode(items, max_len=512, batch=32):
    # bge-m3: dense (CLS, chuẩn hoá) + sparse (lexical weights = ReLU(Linear(hidden)) theo từng token)
    tok = _bgem3_parts(DEVS[0])[0]
    special = np.array(sorted(set(tok.all_special_ids)))
    order = np.argsort([-len(t) for t in items])
    parts = [order[i::len(DEVS)] for i in range(len(DEVS))]
    def run(dev, idx):
        model, lin = _bgem3_parts(dev)[1]
        dense, ids_l, w_l, i, bs = [], [], [], 0, batch
        while i < len(idx):
            part = [items[j] for j in idx[i:i + bs]]
            try:
                enc = tok(part, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
                with torch.inference_mode():
                    hs = model(**enc).last_hidden_state
                    d = torch.nn.functional.normalize(hs[:, 0].float(), dim=-1)
                    w = torch.relu(lin(hs.to(lin.weight.dtype))).squeeze(-1).float()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                if bs == 1: raise
                bs = max(1, bs // 2); print(f"  {dev}: OOM → batch {bs}"); continue
            ids = enc["input_ids"].cpu().numpy(); ww = w.cpu().numpy(); msk = enc["attention_mask"].cpu().numpy()
            for r in range(len(part)):
                keep = (msk[r] == 1) & ~np.isin(ids[r], special)
                ids_l.append(ids[r][keep]); w_l.append(ww[r][keep])
            dense.append(d.cpu().numpy().astype(np.float16)); i += len(part)
        return idx, np.concatenate(dense), ids_l, w_l
    with ThreadPoolExecutor(len(DEVS)) as ex:
        res = [r for r in ex.map(run, DEVS, parts) if len(r[0])]
    D = np.zeros((len(items), res[0][1].shape[1]), np.float16)
    ids_all, w_all = [None] * len(items), [None] * len(items)
    for idx, d, il, wl in res:
        D[idx] = d
        for j, a, b in zip(idx, il, wl):
            ids_all[j], w_all[j] = a, b
    return D, le.weights_to_csr(ids_all, w_all, len(tok))

In [ ]:
def _bgem3():
    from scipy.sparse import save_npz, load_npz
    fE, fS = fp_path(CACHE / "E_bgem3.npy", texts, CFG["BGEM3_MODEL"]), fp_path(CACHE / "S_bgem3.npz", texts, CFG["BGEM3_MODEL"])
    if fE.exists() and fS.exists():
        E, S = np.load(fE), load_npz(fS)
        if len(E) != len(texts) or S.shape[0] != len(texts):      # cache của index khác -> tính lại
            print(f"  cache bge-m3 lệch số dòng ({len(E)}/{S.shape[0]} ≠ {len(texts)}) → tính lại"); E = None
    else:
        E = None
    if E is None:
        E, S = bgem3_encode(texts); np.save(fE, E); save_npz(fS, S)
    bgem3_encode(queries[:8])                                            # nạp model trước, không tính giờ
    t0 = time.time(); QE, QS = bgem3_encode(queries); t_enc = time.time() - t0
    t0 = time.time(); runs["dense_bgem3_base"] = dense_search(E, QE)
    timing["dense_bgem3_base"] = (time.time() - t0 + t_enc / 2) / len(queries) * 1000
    t0 = time.time(); runs["sparse_bgem3"] = le.SparseWeightsIndex(S).search(QS, k=CFG["TOPN_CHUNKS"])
    timing["sparse_bgem3"] = (time.time() - t0 + t_enc / 2) / len(queries) * 1000
    info["dense_bgem3_base"] = dict(desc="BAAI/bge-m3 gốc, dense (CLS) — chưa fine-tune cho tiếng Việt pháp lý")
    info["sparse_bgem3"] = dict(desc="BAAI/bge-m3 learned sparse (trọng số từng token do model học)")
if CFG["RUN_BGEM3"]:
    optional("bge-m3 dense + sparse", _bgem3)

## 7. Biến thể chunk (dense AITeamVN)

In [ ]:
chunk_variant_runs = {"Theo cấu trúc (mặc định)": (runs["dense_aiteamvn"], chunk_keys, len(chunks))}

def _variants():
    c = chunks.sort_values(["record_id", "chunk_index"])
    first = c.drop_duplicates("record_id").set_index("record_id")
    headers = {r: e[: len(e) - len(b)].rstrip() for r, e, b in zip(first.index, first.embed_text, first.body)}
    bodies = c.groupby("record_id", sort=False)["body"].apply(lambda x: " ".join(x))
    rids = bodies.index.tolist()

    # (a) Điều nguyên: 1 vector / Điều (model tự cắt ở 512 token)
    whole = [f"{headers[r]}\n{bodies[r]}" for r in rids]
    E = embed_texts(CFG["EMB_MODEL"], whole, revision=MREV, cache=CACHE / "E_article_whole.npy")
    keys = np.array([rec_key[r] for r in rids], dtype=object)
    chunk_variant_runs["Điều nguyên (cắt 512 token)"] = (dense_search(E, Qv_main), keys, len(rids))

    # (b) Theo từng Khoản: mỗi Khoản 1 chunk (Điều không có Khoản giữ nguyên)
    if lc is None:
        return
    k_texts, k_keys = [], []
    for r in rids:
        row = first.loc[r]
        hdr = lc.build_header(row.subject_title_vi or "", "", row.article_title or "")
        sp = lc.split_khoan(bodies[r])
        if sp:
            pre, items = sp
            h2 = f"{hdr}\n{pre}" if pre and len(pre) < 400 else hdr
            for _, body in items:
                k_texts.append(f"{h2}\n{body}"); k_keys.append(rec_key[r])
        else:
            k_texts.append(f"{hdr}\n{bodies[r]}"); k_keys.append(rec_key[r])
    E = embed_texts(CFG["EMB_MODEL"], k_texts, revision=MREV, cache=CACHE / "E_khoan.npy")
    chunk_variant_runs["Theo từng Khoản"] = (dense_search(E, Qv_main, k=CFG["TOPN_CHUNKS_KHOAN"]),
                                             np.array(k_keys, dtype=object), len(k_texts))

if CFG["RUN_CHUNK_VARIANTS"]:
    optional("biến thể chunk", _variants)
for n, (r, k, nc) in chunk_variant_runs.items():
    print(f"  {n:30s} {nc:8,d} chunk | dev {CFG['PRIMARY']} = {score_articles(to_articles(subset(r, is_dev), k), is_dev):.4f}")

## 8. Hybrid

In [ ]:
lex = [n for n in runs if n.startswith("bm25")]
best_bm25 = max(lex, key=dev_score)
print("BM25 tốt nhất (dev):", best_bm25)
base_ms = timing[best_bm25] + timing["dense_aiteamvn"]   # hai nhánh chạy tuần tự trên một máy -> cộng thời gian

def add_fused(name, fn, ms, desc):
    t0 = time.time(); runs[name] = fn()
    timing[name] = ms + (time.time() - t0) / len(queries) * 1000; info[name] = dict(desc=desc)

if not CFG["LITE"]:
    add_fused("hybrid_rrf_bm25_dense", lambda: le.rrf([runs[best_bm25], runs["dense_aiteamvn"]], 60, CFG["TOPN_CHUNKS"]),
              base_ms, f"RRF (k=60): {best_bm25} + dense_aiteamvn")

alpha_curve = []
for a in CFG["ALPHAS"]:
    r = le.weighted_fusion(subset(runs["dense_aiteamvn"], is_dev), subset(runs[best_bm25], is_dev), a, CFG["TOPN_CHUNKS"],
                           floor=CFG["FUSION_FLOOR"])
    alpha_curve.append(dict(alpha=a, score=float(np.mean([le.per_query_metrics(x, g)[CFG["PRIMARY"]]
                                                           for x, g in zip(to_articles(r), dev_g)]))))
best_alpha = max(alpha_curve, key=lambda d: d["score"])["alpha"]
add_fused(f"hybrid_weighted_a{best_alpha}",
          lambda: le.weighted_fusion(runs["dense_aiteamvn"], runs[best_bm25], best_alpha, CFG["TOPN_CHUNKS"], floor=CFG["FUSION_FLOOR"]),
          base_ms, f"{best_alpha}·dense + {1-best_alpha:.1f}·{best_bm25} (min-max có sàn {CFG['FUSION_FLOOR']}, α dò trên dev)")
if not CFG["LITE"]:
    add_fused(f"hybrid_weighted_a{best_alpha}_nofloor",
              lambda: le.weighted_fusion(runs["dense_aiteamvn"], runs[best_bm25], best_alpha, CFG["TOPN_CHUNKS"], floor=0.0),
              base_ms, f"Như trên nhưng min-max KHÔNG có sàn (để so)")

if "sparse_bgem3" in runs and not CFG["LITE"]:
    add_fused("hybrid_rrf_dense_sparse", lambda: le.rrf([runs["dense_aiteamvn"], runs["sparse_bgem3"]], 60, CFG["TOPN_CHUNKS"]),
              timing["dense_aiteamvn"] + timing["sparse_bgem3"], "RRF: dense_aiteamvn + sparse_bgem3")
    add_fused("hybrid_rrf_3way", lambda: le.rrf([runs[best_bm25], runs["dense_aiteamvn"], runs["sparse_bgem3"]], 60, CFG["TOPN_CHUNKS"]),
              base_ms + timing["sparse_bgem3"], f"RRF 3 đường: {best_bm25} + dense_aiteamvn + sparse_bgem3")
for n in [m for m in runs if m.startswith("hybrid")]:
    print(f"  {n:32s} dev {CFG['PRIMARY']} = {dev_score(n):.4f}")

## 9. Structured — tra thẳng Điều được nhắc trong câu hỏi

In [ ]:
struct_base = max(list(runs), key=dev_score)
def _structured():
    n_with, out = 0, []
    base_arts = to_articles(runs[struct_base])
    t0 = time.time()
    for q, arts in zip(queries, base_arts):
        cited = resolver.gold_keys(q, CFG["REFERENCE_DATE"], corpus_keys)[0]
        n_with += bool(cited); cs = set(cited)
        out.append((cited + [a for a in arts if a not in cs])[:100])
    name = f"citation_lookup+{struct_base}"
    art_runs_direct[name] = out
    timing[name] = timing[struct_base] + (time.time() - t0) / len(queries) * 1000
    info[name] = dict(desc=f"Câu hỏi ghi 'Điều X Luật Y' → đưa Điều đó lên đầu, sau đó là {struct_base}",
                      n_queries_with_citation=int(n_with))
    print(f"Câu hỏi có nhắc Điều cụ thể: {n_with:,}/{len(queries):,} ({n_with/len(queries):.1%})")
if CFG["RUN_STRUCTURED"]:
    optional("structured citation lookup", _structured)

## 10. Rerank top-20 / 50 / 100 (GPU)

In [ ]:
from sentence_transformers import CrossEncoder
rr_base = max([n for n in runs if n.startswith("hybrid")] or ["dense_aiteamvn"], key=dev_score)
RERANKED, rr_mix_rows = None, []
titles = [(t if isinstance(t, str) and t.strip() else q) for t, q in zip(Q["title"], queries)]

def _rerank():
    global RERANKED
    test_ids = np.flatnonzero(is_test)
    if CFG["RERANK_MAX_QUERIES"] and len(test_ids) > CFG["RERANK_MAX_QUERIES"]:
        test_ids = np.sort(np.random.default_rng(CFG["SEED"]).choice(test_ids, CFG["RERANK_MAX_QUERIES"], replace=False))
    dev_ids = np.flatnonzero(is_dev) if CFG["RERANK_ON_DEV"] else np.array([], int)
    ids_all = np.sort(np.concatenate([dev_ids, test_ids])).astype(int)
    base_ix, base_sc = runs[rr_base]
    variants = [("", queries, max(CFG["RERANK_DEPTHS"]))]
    if CFG["RERANK_TITLE"]:
        variants.append(("_title", titles, CFG["RERANK_TITLE_DEPTH"]))
    # Nạp reranker lên từng GPU TRƯỚC khi bấm giờ
    ces = {}
    for dev in DEVS:
        ce = CrossEncoder(CFG["RERANK_MODEL"], device=dev, max_length=CFG["RERANK_MAX_LEN"])
        try: ce.model.half()
        except AttributeError: ce.half()
        ces[dev] = ce
    scores, secs = {}, {}
    for tag, qtexts, D in variants:
        sc_v = {}
        def worker(dev, ids):
            bs = CFG["RERANK_BATCH"]
            for n, i in enumerate(ids):
                cand = np.asarray(base_ix[i][:D]); pairs = [(qtexts[i], texts[int(c)]) for c in cand]
                while True:
                    try:
                        sc_v[i] = np.asarray(ces[dev].predict(pairs, batch_size=bs, show_progress_bar=False), np.float32).ravel(); break
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache(); gc.collect()
                        if bs == 1: raise
                        bs = max(1, bs // 2); print(f"  {dev}: OOM → batch {bs}")
                if n % 250 == 0: print(f"  [{tag or 'câu hỏi'}] {dev}: {n}/{len(ids)}")
        t0 = time.time()
        with ThreadPoolExecutor(len(DEVS)) as ex:
            list(ex.map(worker, DEVS, [ids_all[g::len(DEVS)] for g in range(len(DEVS))]))
        scores[tag], secs[tag] = sc_v, time.time() - t0          # thời gian thực (wall-clock) trên mọi GPU, như các bước khác
        print(f"Rerank{tag or ''} {len(ids_all):,} câu × top-{D}: {secs[tag]/60:.1f} phút")

    def apply(ce_s, d, beta, which):
        ix, sc = list(base_ix), list(base_sc)
        for i in which:
            cand = np.asarray(base_ix[i][:d]); s = np.asarray(ce_s[i][:d], np.float64)
            f = s if beta is None else beta * le._minmax(s) + (1 - beta) * le._minmax(np.asarray(base_sc[i][:d], np.float64))
            o = np.argsort(-f, kind="stable"); rest = np.asarray(base_ix[i][d:])   # stable: điểm bằng nhau giữ thứ tự gốc
            ix[i] = np.concatenate([cand[o], rest]); sc[i] = np.concatenate([f[o], np.full(len(rest), f.min() - 1.0)])
        return ix, sc
    def dev_mean(run):
        arts = to_articles(([run[0][i] for i in dev_ids], [run[1][i] for i in dev_ids]))
        return float(np.mean([le.per_query_metrics(a, gold_l[i])[CFG["PRIMARY"]] for a, i in zip(arts, dev_ids)]))

    for tag, _, D in variants:
        for d in [x for x in CFG["RERANK_DEPTHS"] if x <= D]:
            ms = timing[rr_base] + secs[tag] / len(ids_all) * 1000 * d / D
            name = f"{rr_base}+rerank{d}{tag}"
            runs[name] = apply(scores[tag], d, None, ids_all); timing[name] = ms
            info[name] = dict(desc=f"Vietnamese_Reranker ({'tiêu đề câu hỏi' if tag else 'câu hỏi đầy đủ'}) chấm lại top-{d} chunk của {rr_base}; chỉ dùng điểm reranker")
            if len(dev_ids):
                curve = [(b, dev_mean(apply(scores[tag], d, b, dev_ids))) for b in CFG["RERANK_BETAS"]]
                b_best = max(curve, key=lambda x: x[1])[0]
                rr_mix_rows.append(dict(variant=tag or "full", depth=d, curve=curve, best_beta=b_best))
                mname = f"{name}_mix{b_best}"
                runs[mname] = apply(scores[tag], d, b_best, ids_all); timing[mname] = ms
                info[mname] = dict(desc=f"{b_best}·reranker + {1-b_best:.1f}·{rr_base} (min-max trên top-{d}, β dò trên dev)")
                print(f"  {name}: β tốt nhất trên dev = {b_best} (β=1 là chỉ dùng reranker, β=0 là giữ nguyên hybrid)")
    RERANKED = np.zeros(len(Q), bool); RERANKED[ids_all] = True
if CFG["RUN_RERANK"]:
    print("Rerank trên:", rr_base); optional("rerank", _rerank)

## 11. Kết quả trên tập test

In [ ]:
METRICS = ["ndcg@10", "mrr@10", "recall@10", "recall@100", "hit@1", "hit@10", "p@5", "f2@5", "r_prec", "ap@100"]
if CFG["LITE"]:
    # Chỉ giữ các bậc của "bậc thang"; các run phụ (sparse, rerank trộn câu hỏi đầy đủ…) vẫn tính được nhưng không chấm
    _keep = {"tfidf_syllable", "bm25_syllable", "bm25_syll_bigram", "dense_bgem3_base", "dense_aiteamvn",
             rr_base, f"{rr_base}+rerank20"}
    _keep |= {n for n in runs if n.startswith(f"{rr_base}+rerank20_title_mix")}
    if CFG["LITE_RERANK_ABLATION"]:
        _keep |= {f"{rr_base}+rerank20_title"} | {n for n in runs if n.startswith(f"{rr_base}+rerank20_mix")}
    for _n in [n for n in runs if n not in _keep]:
        runs.pop(_n)
    for _n in [n for n in art_runs_direct if n not in _keep]:
        art_runs_direct.pop(_n)
    print("LITE — chấm", len(runs) + len(art_runs_direct), "phương pháp:", ", ".join(list(runs) + list(art_runs_direct)))
mask = is_test & (RERANKED if RERANKED is not None else True)       # mọi phương pháp chấm trên CÙNG một tập câu
ids = np.flatnonzero(mask)
art_runs = {n: to_articles(r) for n, r in runs.items()}
art_runs.update(art_runs_direct)
per_q = {n: pd.DataFrame([le.per_query_metrics(a[i], gold_l[i]) for i in ids]) for n, a in art_runs.items()}

# Phương án THAM CHIẾU cho kiểm định được chọn trên DEV (không nhìn tập test), bỏ qua quy tắc tra trích dẫn
dev_ids = np.flatnonzero(is_dev)
def dev_primary(n):
    if "+rerank" in n and (RERANKED is None or not RERANKED[dev_ids].all()):
        return float("nan")
    a = art_runs[n]
    return float(np.mean([le.per_query_metrics(a[i], gold_l[i])[CFG["PRIMARY"]] for i in dev_ids]))
dev_scores = {n: dev_primary(n) for n in art_runs}
best = max([n for n in art_runs if not n.startswith("citation_lookup") and not math.isnan(dev_scores[n])], key=dev_scores.get)
print(f"Phương án tham chiếu (tốt nhất trên dev): {best} · dev {CFG['PRIMARY']} = {dev_scores[best]:.4f}")
raw_p = {n: le.paired_randomization(per_q[best][CFG["PRIMARY"]].to_numpy(), per_q[n][CFG["PRIMARY"]].to_numpy(), seed=CFG["SEED"])
         for n in per_q if n != best}
p_holm = le.holm(raw_p)

def family(n):
    return ("structured" if n.startswith("citation_lookup") else "rerank" if "+rerank" in n else "hybrid" if n.startswith("hybrid")
            else "sparse" if n.startswith("sparse") else "dense" if n.startswith("dense") else "lexical")

rows, ci = [], {}
for n, df in per_q.items():
    ci[n] = {m: list(le.bootstrap_ci(df[m].to_numpy(), seed=CFG["SEED"])) for m in ("ndcg@10", "recall@10")}
    rows.append({"method": n, "family": family(n), **df[METRICS].mean().round(4).to_dict(),
                 f"dev_{CFG['PRIMARY']}": round(dev_scores.get(n, float("nan")), 4),
                 "ms/query": round(timing.get(n, float("nan")), 1), "p_holm": p_holm.get(n),
                 "sig": "(best)" if n == best else ("*" if p_holm[n] < 0.05 else "ns"), "desc": info.get(n, {}).get("desc", "")})
res = pd.DataFrame(rows).sort_values(CFG["PRIMARY"], ascending=False).reset_index(drop=True)
res.to_csv(OUT_DIR / "results_test.csv", index=False)
print(f"Tập test: {len(ids):,} câu | tham chiếu = '{best}' (chọn trên dev) | * = kém hơn tham chiếu có ý nghĩa (Holm, p<0.05) | ns = chưa phân biệt được")
res.drop(columns=["desc"])

In [ ]:
lv = Q.loc[mask, "lv_name"].fillna("(không rõ)").to_numpy()
top_lv = pd.Series(lv).value_counts().head(8)
by_lv = pd.DataFrame({n: [per_q[n][CFG["PRIMARY"]][lv == x].mean() for x in top_lv.index] for n in res.method},
                     index=top_lv.index).round(4)
by_lv.to_csv(OUT_DIR / "results_by_domain.csv")

cv_rows = []
for n, (r, k, nc) in chunk_variant_runs.items():
    a = to_articles(r, k); m = pd.DataFrame([le.per_query_metrics(a[i], gold_l[i]) for i in ids])
    cv_rows.append({"name": n, "n_chunks": nc, **m[["ndcg@10", "recall@10", "recall@100", "mrr@10"]].mean().round(4).to_dict()})
chunk_variants = pd.DataFrame(cv_rows); print(chunk_variants.to_string(index=False))

rr_rows = []
if RERANKED is not None:
    for d in [0] + CFG["RERANK_DEPTHS"]:
        n = rr_base if d == 0 else f"{rr_base}+rerank{d}"
        if n in per_q:
            row = {"depth": d, **per_q[n][["ndcg@10", "recall@10", "mrr@10"]].mean().round(4).to_dict()}
            mix = [m for m in per_q if m.startswith(f"{n}_mix")] if d else [n]
            if mix:
                row[f"{CFG['PRIMARY']}_mix"] = round(float(per_q[mix[0]][CFG["PRIMARY"]].mean()), 4)
            rr_rows.append(row)
    print(pd.DataFrame(rr_rows).to_string(index=False))

### Độ nhạy: bỏ các Điều chung chung khỏi đáp án

In [ ]:
# Câu trả lời hay dẫn thêm Điều thủ tục / định nghĩa chung (vd. "Quyền yêu cầu Tòa án bảo vệ", "Giải thích từ ngữ").
# Không xoá hẳn (đôi khi chính là đáp án đúng) mà chấm lại khi bỏ chúng ra, xem thứ hạng phương pháp có đổi không.
gen_re = re.compile(CFG["GENERIC_TITLE_RE"])
generic = {k for k, t in key_title.items() if isinstance(t, str) and gen_re.search(le.norm(t))}
gold_ng = [[k for k in g if k not in generic] for g in gold_l]
keep = [i for i in ids if gold_ng[i]]
n_touch = sum(1 for i in ids if len(gold_ng[i]) < len(gold_l[i]))
print(f"Điều chung chung trong kho: {len(generic):,} · câu test có đáp án chứa chúng: {n_touch:,}/{len(ids):,} · "
      f"câu còn đáp án sau khi bỏ: {len(keep):,}")
top_methods = list(res.method.head(8)) + ([best] if best not in list(res.method.head(8)) else [])
sens = pd.DataFrame([{"method": n, f"{CFG['PRIMARY']} (đáp án đầy đủ)": round(per_q[n][CFG["PRIMARY"]].mean(), 4),
                      f"{CFG['PRIMARY']} (bỏ Điều chung)": round(float(np.mean([le.per_query_metrics(art_runs[n][i], gold_ng[i])[CFG["PRIMARY"]]
                                                                                  for i in keep])), 4)}
                     for n in top_methods])
sens.to_csv(OUT_DIR / "sensitivity_generic.csv", index=False)
print(sens.to_string(index=False))

## 12. Dashboard, lưu kết quả & MLflow

In [ ]:
best_run = art_runs[best]; fails = []
for i in ids:
    m = le.per_query_metrics(best_run[i], gold_l[i])
    if m["hit@10"] == 0:
        fails.append({"item_id": Q.item_id[i], "lv": Q.lv_name[i], "title": Q.title[i],
                      "gold": " ; ".join(key_cite.get(k, k) for k in gold_l[i][:4]), "first_rank": m["first_rank"],
                      "top3": " ; ".join(f"{key_cite.get(k, k)} ({(key_title.get(k) or '')[:40]})" for k in best_run[i][:3])})
fails = pd.DataFrame(fails, columns=["item_id", "lv", "title", "gold", "first_rank", "top3"])
fails.to_csv(OUT_DIR / f"failures_{best}.csv", index=False)

ML = {"ndcg@10": "nDCG@10", "mrr@10": "MRR@10", "recall@10": "Recall@10", "recall@100": "Recall@100", "hit@1": "Hit@1",
      "hit@10": "Hit@10", "p@5": "Precision@5", "f2@5": "F2@5", "r_prec": "R-Precision", "ap@100": "MAP@100"}
data = {
    "meta": {"title": "Benchmark truy xuất văn bản pháp luật Việt Nam",
             "subtitle": f"Corpus: Bộ Pháp điển ({chunks.record_id.nunique():,} Điều) · Bộ đề: Hỏi đáp pháp luật Bộ Tư pháp · metric chính {ML[CFG['PRIMARY']]}",
             "footer": f"{le.EVAL_VERSION} · index model {manifest.get('model', {}).get('id')} @ {str(MREV)[:7]} · {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}",
             "primary": CFG["PRIMARY"], "n_total": int(GOLDEN_STATS.get("questions", len(Q))), "n_gold": int(len(Q)), "n_dev": int(is_dev.sum()),
             "n_test": int(len(ids)), "n_chunks": int(len(chunks)), "n_articles": int(chunks.record_id.nunique()),
             "best": best, "n_fail": int(len(fails)), "rerank_base": rr_base if RERANKED is not None else None},
    "families": [{"id": "lexical", "label": "Lexical"}, {"id": "sparse", "label": "Learned sparse"}, {"id": "dense", "label": "Dense"},
                 {"id": "hybrid", "label": "Hybrid"}, {"id": "rerank", "label": "Rerank"}, {"id": "structured", "label": "Structured"}],
    "metric_labels": ML, "metric_order": METRICS, "table_metrics": METRICS,
    "methods": [{"name": r.method, "family": r.family, "desc": r.desc, "ms": r["ms/query"],
                 "metrics": {m: r[m] for m in METRICS}, "ci": ci[r.method], "p_holm": r.p_holm, "sig": r.sig}
                for _, r in res.iterrows()],
    "ks": list(le.KS),
    "recall_k": {n: [float(per_q[n][f"recall@{k}"].mean()) for k in le.KS] for n in per_q},
    "by_domain": {"domains": [{"name": d, "n": int(c)} for d, c in top_lv.items()],
                  "values": {n: by_lv[n].tolist() for n in by_lv.columns}},
    "alpha": alpha_curve, "rerank_depth": rr_rows, "chunk_variants": chunk_variants.to_dict(orient="records"),
    "citations": [{"reason": r, "n": n} for r, n in GOLDEN_STATS.get("citation_reasons", {}).items()],
    "failures": fails.head(15).to_dict(orient="records"),
    "notes": [
        f"Bộ câu hỏi chuẩn {GOLDEN_SRC} (cùng bộ với cổng chất lượng): đáp án trích từ câu trả lời của cán bộ Bộ Tư pháp, chỉ giữ Điều có nội dung khớp câu trả lời (chế độ 'validated' của mlops.labels)."
        + (f" Ngưỡng lọc tự hiệu chỉnh trên dev: giữ {GOLDEN_STATS['calibration']['tpr']:.0%} trích dẫn có ghi năm, nhận nhầm {GOLDEN_STATS['calibration']['fpr']:.1%}." if GOLDEN_STATS.get("calibration") else ""),
        "Đáp án là nhãn bạc: trích tự động từ câu trả lời của cán bộ Bộ Tư pháp. Câu trả lời hay dẫn thêm Điều thủ tục ít liên quan, nên Recall khó đạt 1.",
        "Câu hỏi gửi từ 2007–2021: chỉ giữ trích dẫn tới văn bản còn trong Bộ Pháp điển và đúng phiên bản theo ngày hỏi; nội dung Điều có thể đã được sửa đổi.",
        "'ns' nghĩa là chưa đủ bằng chứng để nói phương pháp đó kém hơn phương pháp tốt nhất (đã hiệu chỉnh Holm) — khi đó nên chọn phương án nhanh/rẻ hơn.",
        "Recall@100 là trần cho reranker và LLM: Điều đúng không nằm trong top-100 thì bước sau không cứu được.",
        f"Mọi phương pháp được chấm trên cùng {len(ids):,} câu test; tham số (k1, b, qtf, α, β, phương án nền để rerank) và phương án tham chiếu cho kiểm định đều chọn trên tập dev. Bộ lọc đáp án cũng chỉ hiệu chỉnh trên dev.",
        "Độ trễ (ms/câu): thời gian thực trên 2 GPU T4 / CPU của Kaggle, KHÔNG gồm lúc nạp model; hybrid = cộng thời gian hai nhánh (chạy tuần tự). Rerank ở độ sâu nhỏ được ước lượng tỷ lệ theo số đoạn chấm. Dùng để so nhanh–chậm, không phải độ trễ của máy chủ thật.",
        "Precision@5 và F2@5 có trần thấp vì đa số câu chỉ có 1–2 Điều đúng (1 Điều đúng thì P@5 tối đa 0,2); trần này như nhau với mọi phương pháp nên không làm lệch thứ hạng.",
    ],
}
path = dash.render(data, OUT_DIR / "dashboard.html")
print("Dashboard:", path)

In [ ]:
pd.DataFrame([{"item_id": Q.item_id[i], "split": Q.split[i], "method": n, "ranked": list(a[i][:100])}
              for n, a in art_runs.items() for i in range(len(Q))]).to_parquet(OUT_DIR / "runs_top100.parquet", index=False)
pd.concat({n: df.assign(item_id=Q.item_id.to_numpy()[ids]) for n, df in per_q.items()}, names=["method"]) \
  .reset_index(level=0).to_parquet(OUT_DIR / "per_query_test.parquet", index=False)
pd.DataFrame(bm25_grid_log).to_csv(OUT_DIR / "bm25_tuning_dev.csv", index=False)
pd.DataFrame([dict(variant=r["variant"], depth=r["depth"], beta=b, dev_score=v) for r in rr_mix_rows for b, v in r["curve"]]) \
  .to_csv(OUT_DIR / "rerank_beta_dev.csv", index=False)
summary = dict(eval_version=le.EVAL_VERSION, index=manifest, config=CFG, golden=GOLDEN_SRC, golden_stats=GOLDEN_STATS, code_commit=COMMIT,
               n_questions_with_gold=int(len(Q)), n_dev=int(is_dev.sum()), n_scored_test=int(len(ids)),
               best_bm25=best_bm25, best_alpha=best_alpha, rerank_base=rr_base,
               best_method=best, reference_chosen_on="dev", dev_scores=dev_scores, rerank_mix=rr_mix_rows,
               sensitivity_generic=sens.to_dict(orient="records"), info=info, results=res.to_dict(orient="records"))
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
for f in sorted(p for p in OUT_DIR.iterdir() if p.is_file()):
    print(f"{f.name:34s} {f.stat().st_size/1e6:8.2f} MB")
print("\n🏆 Phương pháp tốt nhất:", best, "| mở dashboard.html trong tab Output để xem chi tiết")

In [ ]:
# Ghi MLflow: run cha + mỗi phương pháp một run con (experiment legal-rag-benchmark)
from legal_core.search import Config
from mlops import bench_mlflow
_prod_cfg = Config()                                              # cấu hình production: hybrid α=0,7, BM25 bigram, không rerank
PRODUCTION = f"hybrid_weighted_a{_prod_cfg.alpha}" if f"hybrid_weighted_a{_prod_cfg.alpha}" in set(res.method) else None
if PRODUCTION is None:
    print(f"ℹ️  α dò trên dev = {best_alpha} ≠ α production {_prod_cfg.alpha}: không có dòng nào đúng cấu hình production trong bảng")
BENCH_RUN = bench_mlflow.log_run(
    res=res, per_q=per_q, metrics=METRICS, ks=le.KS, ci=ci, alpha_curve=alpha_curve, rr_mix_rows=rr_mix_rows,
    best=best, production=PRODUCTION, name=f"benchmark {manifest.get('dataset', {}).get('revision', '')[:7] or INDEX_DIR.name} (Kaggle)",
    params={"golden": GOLDEN_SRC, "n_test": int(len(ids)), "n_dev": int(is_dev.sum()), "lite": CFG["LITE"], "best_bm25": best_bm25,
            "best_alpha": best_alpha, "rerank_base": rr_base, "index_model_rev": str(MREV)[:12], "chunks": int(len(chunks))},
    tags={"code_commit": COMMIT, "dataset_rev": str(manifest.get("dataset", {}).get("revision", ""))[:40]},
    out_dir=OUT_DIR, files=("dashboard.html", "results_test.csv", "results_by_domain.csv", "summary.json",
                            f"failures_{best}.csv", "sensitivity_generic.csv"))
print("MLflow run cha:", BENCH_RUN or "(không ghi — thiếu MLFLOW_TRACKING_URI hoặc lỗi, xem cảnh báo)")
from IPython.display import HTML, display
print("Dashboard (cũng nằm trong MLflow: run cha → Artifacts → dashboard.html):")
display(HTML(f'<iframe srcdoc="{__import__("html").escape((OUT_DIR / "dashboard.html").read_text())}" style="width:100%;height:900px;border:0"></iframe>'))

## Đọc kết quả & bước tiếp theo
- **Dashboard:** ngay trên, trong tab *Output* (`benchmark/dashboard.html`), hoặc MLflow → experiment **legal-rag-benchmark** → run cha → Artifacts.
- **So trong MLflow:** mở experiment → tích các run con → biểu tượng **biểu đồ** → chọn `ndcg_at_10`, `hit_at_1`, `ms_per_query`;
  `recall_at_k` hiện thành đường theo k. Run cha có đường `dev_ndcg_vs_alpha` (α×10) và `dev_ndcg_vs_beta/...` (β×10).
- **Đưa phương pháp tốt hơn lên production:** chỉ khi cột `sig` của production là `*` (kém hơn có ý nghĩa) — `ns` thì giữ phương án
  rẻ hơn. Sửa `legal_core/search.py › Config` (α, depth, β) hoặc `values.yaml › rerank.enabled`, push, chạy lại `legal_rag_build`
  → cổng chất lượng chấm cấu hình mới so với champion.
- **Thêm model fine-tune (môn LLMs):** điền `FINETUNED_MODEL`, chạy lại — thành một dòng `dense_finetuned` để so trực tiếp.
- Embedding bge-m3 lưu ở `cache/` — lưu Output này thành Dataset và Add Input lại thì lần sau nhanh hơn.